# LGBM CPU A/B — xsec group-relative prune 11 feats Delta 측정
A: exp_xsec_kaggleA (lgbm_combined, 베이스라인)
B: exp_xsec_kaggleB (lgbm_combined_xsec, +xsec group-relative 11 feats)
동일 커널·동일 환경에서 실행해 클린한 Δ 산출

In [ ]:
# 1) input 자동탐색 (마운트 비표준: /kaggle/input/{datasets,competitions}/...)
import sys, os, glob
from pathlib import Path

print('/kaggle/input:', os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'NONE')

c = glob.glob('/kaggle/input/**/src/config.py', recursive=True)
assert c, 'src/config.py 못 찾음'
SRC_ROOT = str(Path(c[0]).parents[1])
print('SRC_ROOT:', SRC_ROOT)

cc = glob.glob('/kaggle/input/**/playground-series-s6e5', recursive=True)
assert cc, '대회 폴더 못 찾음'
COMP = Path(cc[0])
print('COMP:', COMP)

ac = glob.glob('/kaggle/input/**/f1_strategy_dataset*.csv', recursive=True)
assert ac, '증강 csv 못 찾음'
AUG = Path(ac[0])
print('AUG:', AUG)

print('--- fast-fail 가드 통과 ---')

In [ ]:
# 2) 프로젝트 deps 설치 (CPU — torch/GPU 없음)
import subprocess
def pip(*a):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a], check=True)

pip('lightgbm==4.6.0', 'hydra-core', 'omegaconf', 'python-dotenv')
print('deps 설치 완료')

In [ ]:
# 3) import + 경로 override
import pandas as pd

sys.path.insert(0, SRC_ROOT)
from src import config
from src.train import run
print('import OK:', config.__file__)

config.TRAIN_PATH = COMP / 'train.csv'
config.TEST_PATH = COMP / 'test.csv'
config.SAMPLE_SUBMISSION_PATH = COMP / 'sample_submission.csv'
config.SOURCE_AUG_PATH = AUG

out = Path('/kaggle/working')
config.OOF_DIR = out / 'oof'
config.SUBMISSION_DIR = out / 'submissions'
config.LOG_DIR = out / 'logs'
for d in [config.OOF_DIR, config.SUBMISSION_DIR, config.LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

_a = pd.read_csv(config.SOURCE_AUG_PATH)
print('AUG shape:', _a.shape)
assert len(_a) == 101371, f'증강 행수 불일치: {len(_a)}'
assert config.TRAIN_PATH.exists(), f'train.csv 없음: {config.TRAIN_PATH}'
print('경로 override 완료')

In [ ]:
# 4) A/B 실행 — 동일 커널, 동일 config except features
from omegaconf import OmegaConf
import time, json

CONF = Path(SRC_ROOT) / 'conf'

def run_variant(exp_id, features_conf, notes):
    mc = OmegaConf.load(CONF / 'model' / 'lgbm_tuned.yaml')
    cfg = OmegaConf.create({
        'exp_id': exp_id,
        'notes': notes,
        'use_wandb': False,
        'seed': 42,
        'max_folds': None,
        'kill_criterion': '',
        'model': mc,
        'features': OmegaConf.load(CONF / 'features' / features_conf),
        'augment': {'enabled': True, 'weight': 1.0},
    })
    print(f'\n=== {exp_id} START ===')
    t0 = time.time()
    result = run(cfg)
    dt = time.time() - t0

    log_file = config.LOG_DIR / f'{exp_id}.json'
    best_iters = None
    if log_file.exists():
        log = json.load(open(log_file))
        best_iters = log.get('best_iters', log.get('fold_best_iters'))
        if best_iters:
            capped = [i for i in best_iters if i >= 12000]
            if capped:
                print(f'WARNING [미수렴] best_iter cap(12000) 접촉: fold={capped} -> 미완 학습 가능성')
            else:
                print(f'[수렴 OK] best_iters={best_iters} (모두 < 12000)')

    print(f'{exp_id}: cv_mean={result.get("cv_mean"):.6f} '
          f'folds={[f"{s:.6f}" for s in result.get("fold_scores", [])]} '
          f'best_iters={best_iters} {dt:.0f}s')
    return result

A = run_variant(
    'exp_xsec_kaggleA',
    'lgbm_combined.yaml',
    'Kaggle-A baseline(exp_034) for clean delta'
)
B = run_variant(
    'exp_xsec_kaggleB',
    'lgbm_combined_xsec.yaml',
    'B: +xsec group-relative 11 feats vs Kaggle-A'
)

dA, dB = A.get('cv_mean'), B.get('cv_mean')
print(f'\n=== DELTA = {dB - dA:+.6f}  (A={dA:.6f}  B={dB:.6f}) ===')

In [ ]:
# 5) 산출물 확인
import os
for subdir in ['oof', 'submissions', 'logs']:
    p = Path('/kaggle/working') / subdir
    files = list(p.glob('*')) if p.exists() else []
    print(f'{subdir}/: {[f.name for f in files]}')